---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-33: LangChain <b>Chains</b> Component</h1>

# Learning agenda of this notebook

> **The idea of this lecture:** real tasks need more than one step: fill a prompt, call a model, clean the answer, then maybe call another model with that answer. A **chain** joins these steps with the `|` operator, so that the output of one step becomes the input of the next, and the whole pipeline runs with a single `invoke()`. This way of writing chains is called **LCEL**, the LangChain Expression Language.

1. **Where Chains Fit**
    - What a chain is, and what happened to the old chain classes
    - Setup: the model
2. **Your First Chain**
    - Example 1: `prompt | model | parser`, compared with the same three steps by hand
    - Example 2: A chain is a Runnable too: `batch()` and `stream()`
3. **Output Parsers**
    - Example 3: `PydanticOutputParser`: from the model's text to a Python object
4. **Three Shapes of Chains**
    - Example 4: Sequential: translate, then summarise
    - Example 5: Parallel: two reports at the same time, then one merged report
    - Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback
5. **Making Chains Robust**
    - Example 7: `with_retry()` and `with_fallbacks()`
6. **Where Chains Go Next**
7. **Summary**

> **Note:** all examples use free models: `openai/gpt-oss-120b` on Groq, and a free OpenRouter model as the backup in Example 7.

# <span style='background :lightgreen' >1. Where Chains Fit</span>

In the app of Lec-30, a **chain** is what connects the building blocks: retriever, prompt template and model, one after the other.

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## What is a chain?
- In Lec-31 we filled a template with `invoke()` and gave the result to a model with another `invoke()`. Prompt templates, models and output parsers all have the **same** `invoke()` method: in LangChain they are all **Runnables**.
- Because they share one interface, they can be joined with `|`: `prompt | model | parser` is a new Runnable that calls the three in order and passes each output to the next. This is a **chain**.
- A chain is a Runnable again, so it has the same methods (`invoke`, `batch`, `stream`, …) and can itself be a step of a bigger chain.

## What happened to the old chain classes
Older tutorials build chains with classes. They still work, but they were moved to `langchain-classic` in LangChain 1.0 and print a deprecation warning:

| Old class (`langchain_classic.chains`) | Today, with LCEL (`langchain_core`) |
|---|---|
| `LLMChain(llm=model, prompt=prompt)` | `prompt \| model \| StrOutputParser()` (Example 1) |
| `SimpleSequentialChain`, `SequentialChain` | `chain_1 \| reshape \| chain_2` (Example 4) |
| `RouterChain`, `MultiPromptChain` | `RunnableBranch(...)` (Example 6) |

For workflows with **loops**, saved **state** or several **agents**, LangChain now recommends **LangGraph**, which we meet after the agent lectures. Chains are for fixed pipelines that run from start to end.

> **Documentation:** https://reference.langchain.com/python/langchain-core/runnables

## Setup: the model
The same Groq model as in Lec-30 to 32, created once.

In [1]:
import os                                                   # read GROQ_API_KEY
from dotenv import load_dotenv                              # load the API keys from ../keys/.env
from langchain_openai import ChatOpenAI                     # works with Groq's OpenAI-compatible API (Lec-30, Example 3)

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ (the file wins)
print("GROQ_API_KEY      ", "set" if os.getenv("GROQ_API_KEY") else "MISSING")         # never print any part of a key
print("OPENROUTER_API_KEY", "set" if os.getenv("OPENROUTER_API_KEY") else "MISSING")   # for the backup in Example 7

GROQ = {"base_url": "https://api.groq.com/openai/v1", "api_key": os.getenv("GROQ_API_KEY"), "max_retries": 5}
GROQ_MODEL = ChatOpenAI(model="openai/gpt-oss-120b", **GROQ)   # free; used in every example

GROQ_API_KEY       set
OPENROUTER_API_KEY set


# <span style='background :lightgreen' >2. Your First Chain</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-16.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 1: <code>prompt | model | parser</code>, compared with the same three steps by hand</span>
### Step 1: Three steps by hand
We translate a short biography into Urdu. Each step's output is passed **by our code** to the next step.

In [2]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)

BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. "
                                                "Reply with the translation only.\n\n{text}")

prompt_value = translate_prompt.invoke({"language": "Urdu", "text": BIO})   # step 1: fill the template → PromptValue
ai_message = GROQ_MODEL.invoke(prompt_value)                # step 2: call the model → AIMessage
translation = ai_message.text                               # step 3: keep only the text → str
print(translation)

ڈاکٹر محمد عارف بٹ یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس ڈیپارٹمنٹ میں اسسٹنٹ پروفیسر ہیں۔ اُنہوں نے ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی حاصل کی ہے۔ 33 سے زائد سال کے تدریسی اور انتظامی تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت کی تعلیم دیتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینئر بھی ہیں اور ایکسیلیئٹ اور فالن‑ہنٹ کے بانی ہیں۔


### Step 2: The same three steps as one chain
- `StrOutputParser()` is a small component that takes an `AIMessage` and returns only its text, the same as `.text` in Step 1. As a component, it can be the last step of a chain.
- `translate_prompt | GROQ_MODEL | StrOutputParser()` builds the chain; one `invoke()` runs all three steps.
- `chain.get_graph().print_ascii()` draws the chain: the input, each component, and the output.

In [3]:
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text (a str)

translate_chain = translate_prompt | GROQ_MODEL | StrOutputParser()   # three components, one chain

result = translate_chain.invoke({"language": "Urdu", "text": BIO})    # the same input dict as Step 1
print("Is the result a str?", isinstance(result, str), "\n")   # yes: the parser's output is the chain's output
print(result, "\n")
translate_chain.get_graph().print_ascii()                   # a drawing of the chain (uses the grandalf package)

Is the result a str? True 

ڈاکٹر محمد عارف بٹ ڈیٹا سائنس کے شعبے، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ تعلیم اور انتظامیہ میں 33 سے زائد سال کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت کی تدریس کرتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایگزالیئٹ اور فیلکن-ہنٹ کے بانی ہیں۔ 

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +----

## <span style='color:#7e22ce'>Example 2: A chain is a Runnable too: <code>batch()</code> and <code>stream()</code></span>
The methods of a model (Lec-30, Example 8) work on a whole chain:
- `batch([...])` runs the chain on several inputs at the same time and returns the outputs in the same order.
- `stream(...)` yields the output piece by piece. Because the last step is `StrOutputParser`, the pieces are plain strings.

In [4]:
sentence = "Knowledge is the light that guides us through the darkest times."

inputs = [{"language": language, "text": sentence} for language in ["Urdu", "French", "Arabic"]]   # three inputs
for item, output in zip(inputs, translate_chain.batch(inputs)):   # all three at the same time
    print(f"{item['language']:7} {output}")

print("\nStreamed, piece by piece:")
for piece in translate_chain.stream({"language": "Spanish", "text": BIO}):   # pieces of the answer, as soon as they come
    print(piece, end="", flush=True)                        # end="": no new line; flush=True: show it at once

Urdu    علم وہ روشنی ہے جو ہمیں سب سے تاریک لمحات میں رہنمائی کرتی ہے۔
French  La connaissance est la lumière qui nous guide à travers les moments les plus sombres.
Arabic  المعرفة هي النور الذي يوجهنا عبر أحلك الأوقات.

Streamed, piece by piece:
El Dr. Muhammad Arif Butt es Profesor Asistente en el Departamento de Ciencia de Datos de la Universidad de Punjab, Lahore. Posee una Maestría, una Maestría de Investigación (ambas con medallas de oro) y un doctorado en Ciencias de la Computación. Con más de 33 años de experiencia en la enseñanza y la gestión, imparte cursos de sistemas operativos, programación de sistemas, ciberseguridad e inteligencia artificial. También es emprendedor tecnológico y fundador de Excaliat y Falcon‑Hunt.

# <span style='background :lightgreen' >3. Output Parsers</span>

An **output parser** is the last step of many chains: it turns the model's answer into the type our program needs.

| Parser | Turns the answer into | How |
|---|---|---|
| `StrOutputParser()` | a `str` | takes the text of the `AIMessage` |
| `PydanticOutputParser(pydantic_object=Movie)` | a validated `Movie` object | asks for JSON in the prompt, then reads and checks it |

## <span style='color:#7e22ce'>Example 3: <code>PydanticOutputParser</code>: from the model's text to a Python object</span>
It works in two halves:
1. **Before the model:** `parser.get_format_instructions()` writes instructions that describe the JSON we want, from our Pydantic class. We put them into the prompt with `partial()` (Lec-31, Example 3).
2. **After the model:** the parser reads the JSON in the model's text, checks every field and its type, and returns a `Movie` object. If the JSON is wrong, it raises an `OutputParserException` instead of letting bad data into the program.

In [5]:
from datetime import date                                   # today's date, to compute the film's age
from pydantic import BaseModel, Field                       # describe (and validate) the shape of the answer
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders
from langchain_core.output_parsers import PydanticOutputParser   # the model's JSON text → a Pydantic object

class Movie(BaseModel):                                     # the object we want back
    title: str = Field(description="the film's title")
    year: int = Field(description="the year the film was released")
    director: str = Field(description="the director's full name")

movie_parser = PydanticOutputParser(pydantic_object=Movie)
print(movie_parser.get_format_instructions()[-260:], "\n")  # the end of the instructions: the schema of Movie

movie_prompt = PromptTemplate.from_template("Answer the question.\n{format_instructions}\nQuestion: {question}"
                                            ).partial(format_instructions=movie_parser.get_format_instructions())
movie_chain = movie_prompt | GROQ_MODEL | movie_parser     # the parser is the last step

movie = movie_chain.invoke({"question": "Which film did Francis Ford Coppola release in 1972 about a mafia family?"})
print(type(movie).__name__, "→", movie)                     # a Movie object, not a string
print(f"{movie.title} is {date.today().year - movie.year} years old.")   # the fields can be used directly: year is an int

", "type": "string"}, "year": {"description": "the year the film was released", "title": "Year", "type": "integer"}, "director": {"description": "the director's full name", "title": "Director", "type": "string"}}, "required": ["title", "year", "director"]}
``` 

Movie → title='The Godfather' year=1972 director='Francis Ford Coppola'
The Godfather is 54 years old.


> **`PydanticOutputParser` or `with_structured_output()`?** Both give a validated object. `with_structured_output()` (Lec-30, Example 9) uses the model provider's built-in JSON mode or tool calling, so it is more reliable when the model supports it. `PydanticOutputParser` only needs text, so it works with **any** model, including small local ones, and many RAG tutorials use it.

# <span style='background :lightgreen' >4. Three Shapes of Chains</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-17.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 4: Sequential: translate, then summarise</span>
- We join two chains: `translate_chain` (Example 1) and a new `summarize_chain`. But they do not fit together directly: `translate_chain` returns a **str**, and `summarize_prompt` needs a **dict** with `language` and `text`.
- **`RunnableLambda(function)`** turns any Python function into a Runnable, so it can sit between them with `|`. Here it reshapes the Urdu string into the dict the next prompt expects.
- (A plain function or lambda next to `|` is turned into a `RunnableLambda` automatically; writing it out makes the step easy to see.)

In [6]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text
from langchain_core.runnables import RunnableLambda         # turns a Python function into a chain step

summarize_prompt = PromptTemplate.from_template("Summarise the following {language} text in fewer than 50 words, "
                                                "in {language}:\n\n{text}")
summarize_chain = summarize_prompt | GROQ_MODEL | StrOutputParser()

to_summary_input = RunnableLambda(lambda urdu_text: {"language": "Urdu", "text": urdu_text})   # str → dict

translate_then_summarize = translate_chain | to_summary_input | summarize_chain   # three steps, one chain

print(translate_then_summarize.invoke({"language": "Urdu", "text": BIO}))

ڈاکٹر محمد عارف بٹ، یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس اسسٹنٹ پروفیسر ہیں۔ ایم ایس سی، ایم فل (دونوں گولڈ میڈل) اور پی ایچ ڈی کے حامل، 33 سال کا انتظامی تجربہ، آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور AI پڑھاتے ہیں، اور ایکسیلیئٹ و فالکن‑ہنٹ کے بانی ہیں۔


## <span style='color:#7e22ce'>Example 5: Parallel: two reports at the same time, then one merged report</span>
- **`RunnableParallel({"report_a": chain_a, "report_b": chain_b})`** gives the **same** input to both chains, runs them **at the same time**, and returns one dict with both results under the keys we chose.
- That dict is exactly what the next prompt needs: `{report_a}` and `{report_b}`, so `| merge_chain` follows directly.
- `partial()` (Lec-31) makes two versions of one report template, each with its own perspective fixed, so each needs only the `topic`.
- (A plain dict `{"report_a": chain_a, "report_b": chain_b}` next to `|` means the same as `RunnableParallel(...)`.)
- Running at the same time saves time when the provider serves several requests at once. On Groq's free tier, the limit of about 8,000 tokens per minute can make the second report wait for the first, so the gain varies from minute to minute; with a paid account or a local model, two reports take about as long as one.

In [7]:
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text
from langchain_core.runnables import RunnableParallel       # runs several chains on the same input, at the same time

# "in about 300 words" keeps both reports short enough for the merge step: Groq's free tier rejects
# any single request above 8,000 tokens (HTTP 413), and two long reports together can exceed that
report_prompt = PromptTemplate.from_template("Write a detailed analytical report, in about 300 words, "
                                             "about the {topic} from {perspective}.")
chain_american = report_prompt.partial(perspective="the American perspective") | GROQ_MODEL | StrOutputParser()
chain_middle_eastern = report_prompt.partial(perspective="the Middle Eastern perspective") | GROQ_MODEL | StrOutputParser()

both_reports = RunnableParallel({"report_a": chain_american,          # key → the chain that fills it
                                 "report_b": chain_middle_eastern})

merge_prompt = PromptTemplate.from_template("Combine the following two reports into one unified report "
                                            "of fewer than 50 words.\nReport A: {report_a}\nReport B: {report_b}")
merge_chain = merge_prompt | GROQ_MODEL | StrOutputParser()

final_chain = both_reports | merge_chain                    # {topic} → {report_a, report_b} → one report

print(final_chain.invoke({"topic": "9/11 incident"}), "\n")   # two reports at the same time, then the merge
final_chain.get_graph().print_ascii()

Unified report: On 9/11 al‑Qaeda hijacked four planes, killing ~3,000 and destroying the World Trade Center. The U.S. answered with the Patriot Act, DHS, and wars in Afghanistan/Iraq, reshaping security. Arab states condemned the attacks but viewed U.S. interventions as sovereignty violations, tightening regional counter‑terrorism laws and spawning militant groups. 

        +----------------------------------+         
        | Parallel<report_a,report_b>Input |         
        +----------------------------------+         
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                         

## <span style='color:#7e22ce'>Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback</span>
- **Step 1, classify and keep:** a classifier gives the feedback's sentiment. **`RunnablePassthrough.assign(sentiment=classify)`** passes the input dict **through unchanged** and **adds** a new key with the classifier's result: `{"feedback": …}` becomes `{"feedback": …, "sentiment": "mixed"}`. Without `.assign()`, the next step would receive only the label and the customer's words would be lost.
- The classifier uses `with_structured_output()` (Lec-30, Example 9) with `Literal[...]`, so the label is always exactly one of the three words.
- **Step 2, branch:** **`RunnableBranch((condition, chain), (condition, chain), default_chain)`** checks the conditions in order and runs only the **first** chain whose condition is true; if none is true, it runs the default (here: the mixed reply).
- We use `.assign()` once more to add the reply, so the output shows the feedback, its label and the reply together.

In [8]:
from typing import Literal                                  # a field that may hold only some fixed values
from pydantic import BaseModel                              # describe the classifier's answer
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text
from langchain_core.runnables import RunnableBranch, RunnablePassthrough   # if-else, and "pass the input through"

class Sentiment(BaseModel):                                 # the classifier's answer
    sentiment: Literal["positive", "negative", "mixed"]

classify = (PromptTemplate.from_template("Classify the sentiment of this customer feedback: {feedback}")
            | GROQ_MODEL.with_structured_output(Sentiment)  # → a Sentiment object
            | (lambda result: result.sentiment))            # → just the word (a plain function, used as a step)

def reply_chain(instruction):
    '''A chain that writes a two-sentence reply to the feedback, following the instruction.'''
    return (PromptTemplate.from_template(f"{instruction} Reply in two sentences to this customer feedback: {{feedback}}")
            | GROQ_MODEL | StrOutputParser())

branch = RunnableBranch(
    (lambda x: x["sentiment"] == "positive", reply_chain("Thank the customer warmly.")),   # (condition, chain)
    (lambda x: x["sentiment"] == "negative", reply_chain("Apologise, and say the issue goes to the support team.")),
    reply_chain("Thank the customer for the good points and apologise for the problem."),  # default: mixed
)

feedback_chain = (RunnablePassthrough.assign(sentiment=classify)   # {feedback} → {feedback, sentiment}
                  | RunnablePassthrough.assign(reply=branch))      # → {feedback, sentiment, reply}

feedbacks = ["Excellent service, my order arrived a day early!",
             "The laptop arrived broken and nobody answers my emails.",
             "The delivery was late but the product quality is excellent."]
for result in feedback_chain.batch([{"feedback": f} for f in feedbacks]):   # all three at the same time
    print(f"Feedback:  {result['feedback']}\nSentiment: {result['sentiment']}\nReply:     {result['reply']}\n")

Feedback:  Excellent service, my order arrived a day early!
Sentiment: positive
Reply:     Thank you so much for your kind words—we’re thrilled you loved the service! We’re delighted your order arrived early and look forward to serving you again soon.

Feedback:  The laptop arrived broken and nobody answers my emails.
Sentiment: negative
Reply:     We’re very sorry to hear that your laptop arrived damaged and that you haven’t received a response to your emails. I’ve escalated this issue to our support team, and they will contact you shortly to resolve it.

Feedback:  The delivery was late but the product quality is excellent.
Sentiment: mixed
Reply:     Thank you for sharing your positive experience with the product’s quality; we’re delighted you’re pleased with it. We apologize for the delayed delivery and appreciate your patience.



# <span style='background :lightgreen' >5. Making Chains Robust</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-18.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 7: <code>with_retry()</code> and <code>with_fallbacks()</code></span>
### Step 1: `with_retry()`: try the same step again
- `with_retry(stop_after_attempt=3)` returns a new Runnable that, on an error, waits a little and tries **again**, up to 3 times in all.
- To see it without spending any tokens, we use a small function that fails on its first two calls, as a busy server would.

In [9]:
from langchain_core.runnables import RunnableLambda         # turns a Python function into a chain step

attempts = {"count": 0}                                     # counts the calls

def busy_server(question):
    '''Fails on the first two calls, like a server that is busy for a moment.'''
    attempts["count"] += 1
    print(f"   attempt {attempts['count']} …")
    if attempts["count"] < 3:
        raise ConnectionError("the server is busy")         # a temporary error
    return f"answer to '{question}'"

step = RunnableLambda(busy_server).with_retry(stop_after_attempt=3)   # up to 3 attempts in all
print(step.invoke("What is LCEL?"))

   attempt 1 …
   attempt 2 …
   attempt 3 …
answer to 'What is LCEL?'


### Step 2: `with_fallbacks()`: switch to a backup chain
- `main_chain.with_fallbacks([backup_chain])` runs `main_chain`; if it raises an error, it runs `backup_chain` with the **same** input instead.
- To force an error, the main chain asks Groq for a model that does not exist. The backup chain uses a **free OpenRouter model** (Lec-30, Example 4) with the same prompt.
- In a real app we would combine both: `main_chain.with_retry(stop_after_attempt=3).with_fallbacks([backup_chain])`.

In [10]:
from langchain_openai import ChatOpenAI                     # for Groq
from langchain_openrouter import ChatOpenRouter             # for OpenRouter (Lec-30, Example 4)
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text

greet_prompt = PromptTemplate.from_template("Greet {name} in Urdu, in one line.")

broken_model = ChatOpenAI(model="openai/gpt-oss-999b", **{**GROQ, "max_retries": 0})   # this model does not exist
main_chain = greet_prompt | broken_model | StrOutputParser()

backup_model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free", max_tokens=2000)   # free
backup_chain = greet_prompt | backup_model | StrOutputParser()

try:
    main_chain.invoke({"name": "Arif"})                     # alone, the main chain fails ...
except Exception as error:
    print("Main chain alone:", type(error).__name__, "-", str(error)[:90], "…\n")

robust_chain = main_chain.with_fallbacks([backup_chain])    # ... with a fallback, the backup answers
print("With a fallback: ", robust_chain.invoke({"name": "Arif"}))

Main chain alone: NotFoundError - Error code: 404 - {'error': {'message': 'The model `openai/gpt-oss-999b` does not exist or …

With a fallback:  آداب عارفؔ، آپ کیسے ہیں؟


# <span style='background :lightgreen' >6. Where Chains Go Next</span>

| Where | What the chain does there | Lecture |
|---|---|---|
| **RAG** | `retriever` → `prompt` → `model` → `parser`: answers questions from your own documents, with `RunnableParallel` and `RunnablePassthrough` | Lec-38 |
| **Agents** | An agent is not a fixed chain: the model decides the next step in a loop. We write that loop ourselves, then use `create_agent()` | Lec-39 to 51 |
| **LangGraph** | Workflows with loops, saved state, branches between agents and human approval | A later lecture |

# <span style='background :lightgreen' >7. Summary</span>

| Without chains (Lec-31) | With chains (LCEL) |
|---|---|
| `prompt.invoke()`, then `model.invoke()`, then `.text`, passed along by our code | `prompt \| model \| StrOutputParser()`, one `invoke()` |
| A loop of calls for many inputs | `chain.batch([...])` |
| Asking for JSON and parsing it ourselves | `PydanticOutputParser` as the last step |
| Reshaping data between steps by hand | `RunnableLambda(function)` |
| Running steps one after the other | `RunnableParallel({...})` |
| An `if` around separate calls | `RunnablePassthrough.assign(...)` + `RunnableBranch(...)` |
| `try/except` and our own retry loop | `.with_retry()` and `.with_fallbacks([...])` |

### Key takeaways
- Prompts, models, parsers and functions are all **Runnables**: they share `invoke`, `batch` and `stream`, so they can be joined with `|`.
- A **chain** passes each step's output to the next step, and is itself a Runnable.
- **Output parsers** turn the model's answer into a `str` (`StrOutputParser`) or a validated object (`PydanticOutputParser`).
- Three shapes cover most pipelines: **sequential** (`|`, with `RunnableLambda` to reshape data), **parallel** (`RunnableParallel`), and **branching** (`RunnableBranch`, with `.assign()` to keep the input).
- `with_retry()` survives short failures and `with_fallbacks()` switches to a backup: both matter with free models.
- The old `LLMChain`-style classes are deprecated; loops and state belong to LangGraph.
- **Next lecture (Lec-34):** **Document loaders**: getting your own documents into LangChain.

### Check your understanding
1. What do prompt templates, models and output parsers have in common that lets them be joined with `|`?
2. In Example 1, which line of Step 1 does `StrOutputParser()` replace?
3. What does `translate_chain.batch(inputs)` return, and in which order?
4. What are the two halves of `PydanticOutputParser`'s work, before and after the model? What happens if the model's JSON is wrong?
5. Why could `translate_chain` and `summarize_chain` not be joined directly in Example 4? What did `RunnableLambda` do?
6. What does `RunnableParallel` return, and why does that fit `merge_prompt`? When does running in parallel save time, and when not?
7. In Example 6, what would the reply prompt receive without `RunnablePassthrough.assign()`?
8. How does `RunnableBranch` choose a branch, and what happens when no condition is true?
9. When would you use `with_retry()`, and when `with_fallbacks()`?
10. Why does LangChain recommend LangGraph instead of chains for workflows with loops?